In [ ]:
"""Create list of already done jobs to exclude from rerun"""

import csv
import hashlib
import sys
from pathlib import Path
import json

# Load whitelist from JSON file
with open("RUNKEY_WHITELIST.json", "r") as f:
    WHITELIST = json.load(f)["whitelist"]

# CSV files passed as arguments
input_files = [
    "/Users/julian_dev/Projects/MasterThesis/QAOA/logs/ADAM/qaoa_results_ADAM_20260408_182901.csv",
    "/Users/julian_dev/Projects/MasterThesis/QAOA/logs/ADAM/qaoa_results_ADAM_20260408_151931.csv"
]
output_file = "completed_runs.txt"

def normalize(value):
    if value is None:
        return ""
    try:
        return json.dumps(value, sort_keys=True)
    except TypeError:
        return str(value).strip()

def build_run_key(row):
    """Create deterministic hash from whitelist params."""
    key_string = "|".join(
        f"{k}={normalize(row.get(k, ''))}" for k in WHITELIST
    )
    return hashlib.sha1(key_string.encode()).hexdigest()

seen = set()

for file in input_files:
    file = Path(file)
    if not file.exists():
        continue

    with open(file, newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            # Skip incomplete rows
            if "result" not in row or row["result"] in ("", "None"):
                continue

            run_key = build_run_key(row)
            print("PY:", "|".join(f"{k}={normalize(row.get(k, ''))}" for k in WHITELIST))
            seen.add(run_key)

# Write output
with open(output_file, "w") as f:
    for key in sorted(seen):
        f.write(key + "\n")

print(f"Collected {len(seen)} completed runs into {output_file}")

Collected 133 completed runs into completed_runs.txt
